# Baseline BERTimbau — RECLin-PT (Colab T4)

Treina o baseline de extração de relação em 3 classes (`negation_of`, `associated_with`, `no_relation`) sobre os splits do SemClinBr usando o **BERTimbau** (`neuralmind/bert-base-portuguese-cased`), encoder **geral (pré-treinado no brWaC)** do português.

É o **par de comparação** do baseline BioBERTpt (encoder **clínico**). Mesma representação de entrada, loss, scheduler, early stopping, salvamento, métricas, seed e logging — a **única** diferença é o checkpoint de pré-treino. Assim respondemos, de forma válida:

> **"O pré-treinamento clínico importa para extração de relações em textos médicos em português?"**

Métricas: Macro/Micro/Weighted-F1, **MCC**, F1 por classe, matriz de confusão e curvas de treino/validação. O **teste de significância** entre os dois modelos está no notebook `03_significance_colab.ipynb` (rode depois de treinar os dois).

Antes de rodar: **Ambiente de execução → Alterar o tipo de ambiente → GPU (T4)** e cadastre o secret `GITHUB_PAT` (ícone de chave 🔑).

O treino salva **`best_model/` + `last_checkpoint/` por época no Google Drive** e **retoma automaticamente** se o runtime cair.

## 1. Clonar o repositório

In [ ]:
import os, subprocess
from google.colab import userdata

TOKEN = userdata.get('GITHUB_PAT')           # secret do Colab, nunca hard-coded
REPO_NAME = 'RECLin-PT'                       # nome do repo no GitHub
REPO = f'/content/{REPO_NAME}'
GIT_USER_NAME = 'angeloalsf'
GIT_USER_EMAIL = 'angeloalsf@gmail.com'

url = f'https://{TOKEN}@github.com/angeloalsf/{REPO_NAME}'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', url, REPO], check=True)
else:
    print('Repo já clonado; fazendo pull --rebase para integrar mudanças remotas.')
    subprocess.run(['git', '-C', REPO, 'pull', '--rebase', '--autostash'], check=False)

subprocess.run(['git', '-C', REPO, 'config', 'user.name', GIT_USER_NAME], check=True)
subprocess.run(['git', '-C', REPO, 'config', 'user.email', GIT_USER_EMAIL], check=True)
subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', url], check=True)
print('Repo pronto em', REPO)


## 2. Verificar a GPU (T4)

In [ ]:
!nvidia-smi
import torch
print('CUDA disponível:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '—')
assert torch.cuda.is_available(), 'Sem GPU! Ative em Ambiente de execução → GPU (T4).'


## 3. Montar o Google Drive (checkpoints)

Checkpoints em `MyDrive/RECLin-PT/checkpoints_bertimbau_seed42/` — **pasta própria** deste modelo. Dentro: `best_model/` (pesos do melhor epoch) e `last_checkpoint/` (estado de retomada).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = f'/content/drive/MyDrive/{REPO_NAME}/checkpoints_bertimbau_seed42'
os.makedirs(CKPT_DIR, exist_ok=True)
print('Checkpoints em:', CKPT_DIR)
existe = os.path.isfile(os.path.join(CKPT_DIR, 'last_checkpoint', 'training_state.pt'))
print('last_checkpoint existente?', existe, '(se True, o treino vai RETOMAR de onde parou)')


## 4. Instalar dependências

Fonte única: `requirements.txt` do repo — a mesma lista usada fora do Colab. O `torch` com CUDA que o Colab já traz satisfaz `torch>=2.2`, então o `pip` não o reinstala.

In [ ]:
!pip install -q -r $REPO/requirements.txt


## 5. Conferir os splits

Os splits (`data/splits/*.jsonl`) são versionados no repo (seed 42, nível de documento) — **os mesmos para os dois modelos**.

In [ ]:
import subprocess, sys
splits = os.path.join(REPO, 'data', 'splits')
need = ['train.jsonl', 'dev.jsonl', 'test.jsonl']
have = all(os.path.isfile(os.path.join(splits, f)) for f in need)

if not have:
    xml = os.path.join(REPO, 'SemClinBr-xml-public-v1')
    if os.path.isdir(xml) and os.listdir(xml):
        print('Splits ausentes; gerando a partir dos XMLs...')
        subprocess.run([sys.executable, 'src/parse_semclinbr.py',
                        '--xml-dir', 'SemClinBr-xml-public-v1',
                        '--out', 'data/processed/dataset.jsonl'], cwd=REPO, check=True)
        subprocess.run([sys.executable, 'src/make_splits.py'], cwd=REPO, check=True)
    else:
        raise FileNotFoundError('Splits não encontrados e XMLs ausentes.')

for f in need:
    n = sum(1 for _ in open(os.path.join(splits, f), encoding='utf-8'))
    print(f'{f}: {n} docs')


## 6. Token do Hugging Face Hub (Colab Secrets)

Liga o **backup automático** do `best_model/` num repositório **privado** do HF Hub, a cada nova melhor época. É camada **extra**: o Drive continua sendo o destino principal. Serve para o caso de o runtime cair na época seguinte — ou de a cota do Drive estourar.

O token nunca é colado em texto plano aqui; vem dos **Secrets** do Colab.

1. Crie um Access Token do tipo **Write** em <https://huggingface.co/settings/tokens>
2. No Colab: ícone de **chave** no painel esquerdo → **+ Add new secret** → nome `HF_TOKEN` → cole o valor → ligue **Notebook access**
3. Rode a célula abaixo

Sem o secret o treino roda igual; só o backup extra fica de fora.

In [ ]:
import os, re
from google.colab import userdata

HF_USER = 'angeloalsf'
MODELO = 'bertimbau'

# A semente sai do proprio CKPT_DIR (.../checkpoints_bertimbau_seed42), que ja e o que
# voce edita a cada execucao -- assim o repo de backup nunca aponta para a
# semente errada por esquecimento.
_m = re.search(r'seed(\d+)$', CKPT_DIR)
SEED = int(_m.group(1)) if _m else None
HF_BACKUP_REPO = f'{HF_USER}/reclin-pt-{MODELO}-seed{SEED}' if SEED else None

try:
    hf_token = userdata.get('HF_TOKEN')
except Exception as erro:
    hf_token = None
    print('Secret HF_TOKEN indisponivel (%s).' % type(erro).__name__)

if hf_token and HF_BACKUP_REPO:
    # O token viaja SO pelo ambiente: o processo de treino herda os.environ, e
    # src/hf_backup.py le $HF_TOKEN quando --hf-token nao e passado. Assim o
    # valor nunca entra no argv nem no proprio HF_FLAGS.
    os.environ['HF_TOKEN'] = hf_token
    HF_FLAGS = f'--hf-backup-repo {HF_BACKUP_REPO}'
    print('HF_TOKEN carregado dos Secrets do Colab e exportado para o ambiente')
    print('(o valor nunca e impresso nem passa pela linha de comando).')
    print('Backup automatico ->', HF_BACKUP_REPO, '(repositorio PRIVADO)')
    print('Semente lida do CKPT_DIR:', SEED,
          '-- confira que bate com --seed na celula de treino.')
else:
    HF_FLAGS = ''
    print('BACKUP NO HF HUB DESLIGADO nesta execucao.')
    if not HF_BACKUP_REPO:
        print('  Motivo: nao consegui ler a semente de CKPT_DIR =', CKPT_DIR)
    else:
        print('  Motivo: secret HF_TOKEN ausente ou sem "Notebook access".')
        print('  1. Crie um token WRITE em https://huggingface.co/settings/tokens')
        print('  2. Colab: icone de chave (Secrets) no painel esquerdo >')
        print('     "+ Add new secret" > nome HF_TOKEN > cole o valor >')
        print('     ligue "Notebook access"')
        print('  3. Rode esta celula de novo.')
    print('  O treino roda normalmente; so o backup extra fica de fora.')

## 7. Treinar o baseline BERTimbau (com retomada)

Encoder `neuralmind/bert-base-portuguese-cased`, marcadores `[E1] [/E1] [E2] [/E2]`, 3 classes, `class_weight=balanced`. Melhor época pelo macro-F1 no **dev**; **test** uma única vez.

**Hiperparâmetros idênticos entre os dois modelos.** Se o runtime cair, **rode esta célula de novo**: ele pula as épocas já feitas.

Multi-seed manual: mude `--seed`, e use `--out results/baseline_bertimbau_seed43.json` e um `--ckpt-dir` com sufixo `_seed43`.

> ⚠️ **`max_gap` = 25 (era 20).** Mudou após a análise de sensibilidade em `analysis/max_gap/`: a janela de 20 caracteres descartava 1.324 relações anotadas (1.272 delas `associated_with`) antes de virarem candidato. **Os quatro experimentos (2 modelos × 2 sementes) precisam ser refeitos do zero com 25** — apague/renomeie o `--ckpt-dir` antigo antes de rodar, senão a retomada carrega um checkpoint de `max_gap=20` e a config não bate.

In [ ]:
%cd $REPO
!python -u src/baseline_bertimbau.py \
    --splits-dir data/splits \
    --model neuralmind/bert-base-portuguese-cased \
    --epochs 3 \
    --batch-size 64 \
    --max-gap 25 \
    --max-length 128 \
    --seed 42 \
    --ckpt-dir "$CKPT_DIR" \
    $HF_FLAGS \
    --out results/baseline_bertimbau_seed42.json


## 8. Resultados (métricas + matriz de confusão)

In [ ]:
import json
r = json.load(open(os.path.join(REPO, 'results', 'baseline_bertimbau_seed42.json'), encoding='utf-8'))
print('Modelo:', r['model'], '| parâmetros:', f"{r['n_params']/1e6:.1f}M")
print('Candidatos:', r['n_candidates'])
print('Macro-F1 :', round(r['test_macro_f1'], 4))
print('Micro-F1 :', round(r['test_micro_f1'], 4))
print('Weighted :', round(r['test_weighted_f1'], 4))
print('MCC      :', round(r['test_mcc'], 4))
print('\nF1 por classe:')
for k, v in r['test_f1_per_class'].items():
    mark = '   <<< NEGAÇÃO' if k == 'negation_of' else ''
    print(f'  {k:<18s} {v:.4f}{mark}')

cm = r['confusion_matrix']; labels = cm['labels']
print('\nMatriz de confusão (linhas=verdadeiro, colunas=predito):')
print('true\\pred'.ljust(12) + ''.join(s[:8].rjust(10) for s in labels))
for i, l in enumerate(labels):
    print(l[:11].ljust(12) + ''.join(str(int(c)).rjust(10) for c in cm['matrix'][i]))


## 9. Curvas de treino/validação

De `dev_history`: loss de treino vs. validação (overfitting) e F1 no dev por época (seleção de época).

In [ ]:
import matplotlib.pyplot as plt
h = r['dev_history']
ep = [e['epoch'] for e in h]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, [e['train_loss'] for e in h], 'o-', label='train_loss')
ax[0].plot(ep, [e.get('dev_loss') for e in h], 's-', label='dev_loss')
ax[0].set_xlabel('época'); ax[0].set_ylabel('loss'); ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, [e['dev_macro_f1'] for e in h], 'o-', label='dev_macro_f1')
ax[1].plot(ep, [e['dev_negation_of_f1'] for e in h], 's-', label='dev_negation_of_f1')
ax[1].set_xlabel('época'); ax[1].set_ylabel('F1'); ax[1].set_title('F1 no dev'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


## 10. Salvar resultados + predições no GitHub

Forçamos o `add` (o `-f` é histórico: `results/` já não é gitignored) do JSON de métricas, do `.preds.json` (predições do test, usadas no notebook de significância), do `.dev_preds.json` (predições do dev na melhor época, base de qualquer calibração pós-hoc) e do `.test_evals.jsonl` (auditoria de multiplicidade). Sem isso os dois últimos morrem com o runtime.

In [ ]:
import subprocess
for f in ['results/baseline_bertimbau_seed42.json',
          'results/baseline_bertimbau_seed42.preds.json',
          'results/baseline_bertimbau_seed42.dev_preds.json',
          'results/baseline_bertimbau_seed42.test_evals.jsonl']:
    subprocess.run(['git', '-C', REPO, 'add', '-f', f], check=True)
subprocess.run(['git', '-C', REPO, 'commit', '-m',
                'resultados: baseline BERTimbau (Colab T4)'], check=False)
subprocess.run(['git', '-C', REPO, 'pull', '--rebase', '--autostash'], check=False)
subprocess.run(['git', '-C', REPO, 'push'], check=False)
print('Push concluído (se houve mudanças).')


## 11. (Reserva) Reenviar o `best_model/` para o HF Hub à mão

Com a seção 6 configurada, o `best_model/` já sobe **sozinho** a cada nova melhor época, durante o treino. Esta célula é a **rede de segurança**: use se o log tiver mostrado `Backup HF: envio ABANDONADO` (a rede caiu nas três tentativas), ou para reenviar um checkpoint antigo.

Usa a mesma função do pipeline (`src/hf_backup.py`), então o repositório sai **privado** igual — a versão anterior desta célula criava um repo **público**.

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO, 'src'))
from hf_backup import upload_best_model, resolve_token

best = os.path.join(CKPT_DIR, 'best_model')
token, origem = resolve_token()          # le de $HF_TOKEN, definido na secao 6
print('token:', ('presente (via %s)' % origem) if token else 'AUSENTE')

upload_best_model(best, HF_BACKUP_REPO, token,
                  commit_message='reenvio manual do best_model')
print('best_model em https://huggingface.co/' + HF_BACKUP_REPO)